# Project Overview 

### The Revenue Growth & Personalization Analytics Package is an advanced data analytics system designed to help businesses improve sales performance, customer retention, and personalized marketing strategies.

### Modern businesses generate large volumes of transactional and customer behavior data. However, many companies fail to utilize this data effectively for decision-making. This project aims to transform raw sales and customer data into actionable business insights and automated recommendation systems.

In [1]:
# Library Imports and Initial Setup

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Data processing
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Recommendation algorithms
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

# Clustering
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Forecasting
import xgboost as xgb

# For progress tracking
from tqdm import tqdm


####  Load And Explore Data

In [3]:
# Load the dataset
df = pd.read_csv(r"C:\Users\riyac\OneDrive\Desktop\Adler internship\product sale data task.csv")
df

,customer_name,customer_id,product_name,order_id,order_date,city,category,price,total,unit_price,discount,quantity
0,Aarav Singh,C100000,Charger,O500000,22-05-2023,Mumbai,Electronics,100684,100684.00,25171,0,4
1,Aarav Deshmukh,C100001,Backpack,O500001,23-06-2023,Bengaluru,Accessories,69456,65983.20,17364,5,4
2,Sahil Chavan,C100002,Power Bank,O500002,22-09-2023,Kolkata,Electronics,273065,259411.75,54613,5,5
3,Riya Sharma,C100003,Charger,O500003,26-07-2023,Hyderabad,Electronics,25338,24071.10,25338,5,1
4,Karan Sharma,C100004,Headphones,O500004,13-07-2023,Kolkata,Electronics,38792,32973.20,38792,15,1
...,...,...,...,...,...,...,...,...,...,...,...,...
1995,Meera Iyer,C101995,Smartphone,O501995,08-03-2023,Bengaluru,Electronics,171572,162993.40,42893,5,4
1996,Karan Patil,C101996,Tablet,O501996,08-02-2023,Ahmedabad,Electronics,55926,50333.40,27963,10,2
1997,Simran Patil,C101997,Camera,O501997,11-03-2023,Kolkata,Electronics,186425,158461.25,37285,15,5
1998,Simran Patil,C101998,Keyboard,O501998,13-08-2023,Hyderabad,Electronics,17502,16626.90,8751,5,2


In [4]:
print(f"Dataset shape: {df.shape}")
print(f"\nFirst 5 rows of the dataset:")
print(df.head())

Dataset shape: (2000, 12)

First 5 rows of the dataset:
    customer_name customer_id product_name order_id  order_date       city  \
0     Aarav Singh     C100000      Charger  O500000  22-05-2023     Mumbai   
1  Aarav Deshmukh     C100001     Backpack  O500001  23-06-2023  Bengaluru   
2    Sahil Chavan     C100002   Power Bank  O500002  22-09-2023    Kolkata   
3     Riya Sharma     C100003      Charger  O500003  26-07-2023  Hyderabad   
4    Karan Sharma     C100004   Headphones  O500004  13-07-2023    Kolkata   

      category   price      total  unit_price  discount  quantity  
0  Electronics  100684  100684.00       25171         0         4  
1  Accessories   69456   65983.20       17364         5         4  
2  Electronics  273065  259411.75       54613         5         5  
3  Electronics   25338   24071.10       25338         5         1  
4  Electronics   38792   32973.20       38792        15         1  


In [5]:
print(f"\nDataset information:")
print(df.info())


Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   customer_name  2000 non-null   object 
 1   customer_id    2000 non-null   object 
 2   product_name   2000 non-null   object 
 3   order_id       2000 non-null   object 
 4   order_date     2000 non-null   object 
 5   city           2000 non-null   object 
 6   category       2000 non-null   object 
 7   price          2000 non-null   int64  
 8   total          2000 non-null   float64
 9   unit_price     2000 non-null   int64  
 10  discount       2000 non-null   int64  
 11  quantity       2000 non-null   int64  
dtypes: float64(1), int64(4), object(7)
memory usage: 187.6+ KB
None


In [6]:
print(f"\nBasic statistics of numerical columns:")
print(df.describe())


Basic statistics of numerical columns:
               price          total    unit_price     discount     quantity
count    2000.000000    2000.000000   2000.000000  2000.000000  2000.000000
mean   121397.329500  109229.477825  39926.022000    10.067500     3.023000
std     95581.113881   86689.226346  22895.066076     7.057468     1.411193
min       318.000000     254.400000    283.000000     0.000000     1.000000
25%     43308.000000   38969.375000  19828.750000     5.000000     2.000000
50%     95639.000000   86092.850000  39265.000000    10.000000     3.000000
75%    179706.000000  160067.812500  60182.000000    15.000000     4.000000
max    398980.000000  396210.000000  79924.000000    20.000000     5.000000


In [7]:
print(f"\nMissing values per column:")
print(df.isnull().sum())


Missing values per column:
customer_name    0
customer_id      0
product_name     0
order_id         0
order_date       0
city             0
category         0
price            0
total            0
unit_price       0
discount         0
quantity         0
dtype: int64


### Normalize Data Formats

In [9]:
print("Normalizing data formats...")

# Convert order_date to datetime
df['order_date'] = pd.to_datetime(df['order_date'], format='%d-%m-%Y', errors='coerce')
print(f"Date range: {df['order_date'].min()} to {df['order_date'].max()}")

Normalizing data formats...
Date range: 2023-01-01 00:00:00 to 2024-01-01 00:00:00


In [10]:
# Ensure numeric columns are proper type
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df['total'] = pd.to_numeric(df['total'], errors='coerce')
df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')
df['discount'] = pd.to_numeric(df['discount'], errors='coerce')
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')

In [11]:
# Verify data types after conversion
print("\nData types after normalization:")
print(df.dtypes)


Data types after normalization:
customer_name            object
customer_id              object
product_name             object
order_id                 object
order_date       datetime64[ns]
city                     object
category                 object
price                     int64
total                   float64
unit_price                int64
discount                  int64
quantity                  int64
dtype: object


In [12]:
# Save cleaned data
df_cleaned = df.copy()
df_cleaned.to_csv('cleaned_sales_data.csv', index=False)
print("\nCleaned data saved to 'cleaned_sales_data.csv'")


Cleaned data saved to 'cleaned_sales_data.csv'


###  Feature Engineering - Customer Features (RFM)

In [24]:
# Reference date for recency calculation (day after last purchase)
reference_date = df['order_date'].max() + timedelta(days=1)

# Calculate RFM metrics
customer_rfm = df.groupby('customer_id').agg({
    'order_date': lambda x: (reference_date - x.max()).days,  # Recency
    'order_id': 'count',  # Frequency
    'total': 'sum'  # Monetary
}).reset_index()

customer_rfm.columns = ['customer_id', 'recency', 'frequency', 'monetary']

In [26]:
# Add customer lifetime value metrics
customer_stats = df.groupby('customer_id').agg({
    'quantity': 'mean',
    'discount': 'mean',
    'unit_price': 'mean'
}).reset_index()
customer_stats.columns = ['customer_id', 'avg_quantity', 'avg_discount', 'avg_unit_price']

In [28]:
# Merge all customer features
customer_features = customer_rfm.merge(customer_stats, on='customer_id')

print(f"Created features for {len(customer_features)} customers")
print("\nSample of customer features:")
print(customer_features.head())

Created features for 2000 customers

Sample of customer features:
  customer_id  recency  frequency   monetary  avg_quantity  avg_discount  \
0     C100000      225          1  100684.00           4.0           0.0   
1     C100001      193          1   65983.20           4.0           5.0   
2     C100002      102          1  259411.75           5.0           5.0   
3     C100003      160          1   24071.10           1.0           5.0   
4     C100004      173          1   32973.20           1.0          15.0   

   avg_unit_price  
0         25171.0  
1         17364.0  
2         54613.0  
3         25338.0  
4         38792.0  


In [32]:
# Save customer features
customer_features.to_csv('customer_features.csv', index=False)
print("\nCustomer features saved to 'customer_features.csv'")


Customer features saved to 'customer_features.csv'


### Frature Engineering - Product Features

In [35]:
# Calculate product popularity and other metrics
product_features = df.groupby(['product_name', 'category']).agg({
    'order_id': 'count',  # Popularity (number of orders)
    'quantity': 'sum',  # Total units sold
    'total': 'sum',  # Total revenue
    'unit_price': 'mean',  # Average price
    'discount': 'mean'  # Average discount
}).reset_index()

product_features.columns = ['product_name', 'category', 'popularity', 'total_units_sold', 
                           'total_revenue', 'avg_price', 'avg_discount']

In [37]:
# Create price bands with clear, non-overlapping labels
price_bins = [0, 10000, 50000, 100000, 500000, float('inf')]
price_labels = ['Budget (0-10k)', 'Economy (10k-50k)', 'Standard (50k-100k)', 
                'Premium (100k-500k)', 'Luxury (500k+)']

product_features['price_band'] = pd.cut(
    product_features['avg_price'], 
    bins=price_bins,
    labels=price_labels
)

# Optional: Add price band description
product_features['price_range'] = product_features['price_band'].astype(str)

print(f"Price bands created with ranges: {price_labels}")

Price bands created with ranges: ['Budget (0-10k)', 'Economy (10k-50k)', 'Standard (50k-100k)', 'Premium (100k-500k)', 'Luxury (500k+)']


In [39]:
# Calculate popularity score (normalized between 0 and 1)
product_features['popularity_score'] = (product_features['popularity'] - product_features['popularity'].min()) / \
                                        (product_features['popularity'].max() - product_features['popularity'].min())

print(f"Created features for {len(product_features)} products")
print("\nSample of product features:")
print(product_features.head())

Created features for 15 products

Sample of product features:
  product_name     category  popularity  total_units_sold  total_revenue  \
0     Backpack  Accessories         139               388    14038257.85   
1       Camera  Electronics         116               351    12965627.10   
2      Charger  Electronics         139               451    16316790.80   
3   Headphones  Electronics         137               409    13911314.55   
4        Jeans      Fashion         137               425    15066470.35   

      avg_price  avg_discount         price_band        price_range  \
0  39817.827338     10.287770  Economy (10k-50k)  Economy (10k-50k)   
1  40457.568966     10.431034  Economy (10k-50k)  Economy (10k-50k)   
2  40528.669065     11.007194  Economy (10k-50k)  Economy (10k-50k)   
3  37753.197080      9.890511  Economy (10k-50k)  Economy (10k-50k)   
4  39171.927007     10.839416  Economy (10k-50k)  Economy (10k-50k)   

   popularity_score  
0          0.777778  
1         

In [41]:
# Save product features
product_features.to_csv('product_features.csv', index=False)
print("\nProduct features saved to 'product_features.csv'")


Product features saved to 'product_features.csv'


### Feature Engineering - User-Product Interaction Matrix

In [44]:
# Create interaction matrix based on purchase frequency
interaction_matrix = df.groupby(['customer_id', 'product_name']).size().reset_index(name='purchase_count')
interaction_matrix['rating_implicit'] = interaction_matrix['purchase_count'].clip(upper=5)  # Implicit rating

print(f"Interaction matrix shape: {interaction_matrix.shape}")
print(f"Unique users: {interaction_matrix['customer_id'].nunique()}")
print(f"Unique products: {interaction_matrix['product_name'].nunique()}")

print("\nSample of interaction matrix:")
print(interaction_matrix.head(10))

Interaction matrix shape: (2000, 4)
Unique users: 2000
Unique products: 15

Sample of interaction matrix:
  customer_id product_name  purchase_count  rating_implicit
0     C100000      Charger               1                1
1     C100001     Backpack               1                1
2     C100002   Power Bank               1                1
3     C100003      Charger               1                1
4     C100004   Headphones               1                1
5     C100005       Tablet               1                1
6     C100006   Headphones               1                1
7     C100007   Smartphone               1                1
8     C100008   Power Bank               1                1
9     C100009   Smartphone               1                1


In [46]:
# Save interaction matrix
interaction_matrix.to_csv('interaction_matrix.csv', index=False)
print("\n Interaction matrix saved to 'interaction_matrix.csv'")
print("\n STEP 1 COMPLETED: All feature engineering done")


 Interaction matrix saved to 'interaction_matrix.csv'

 STEP 1 COMPLETED: All feature engineering done


### Recommendation Engine - Content-Based Filtering

In [49]:
# Create product feature matrix for content-based filtering
product_content = product_features.copy()
product_content['product_text'] = product_content['product_name'] + ' ' + \
                                  product_content['category'] + ' ' + \
                                  product_content['price_band'].astype(str)

In [51]:
# Create TF-IDF vectors for product descriptions
tfidf = TfidfVectorizer(stop_words='english', max_features=100)
tfidf_matrix = tfidf.fit_transform(product_content['product_text'])

In [53]:
# Calculate cosine similarity between products
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [55]:
# Create a mapping of product names to indices
product_indices = pd.Series(product_content.index, index=product_content['product_name']).drop_duplicates()

def get_content_based_recommendations(product_name, n_recommendations=5):
    """Get content-based recommendations for a product"""
    if product_name not in product_indices:
        return []
    
    idx = product_indices[product_name]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:n_recommendations+1]
    product_indices_rec = [i[0] for i in sim_scores]
    return product_content.iloc[product_indices_rec]['product_name'].tolist()

In [57]:
# Test content-based recommendation
test_product = product_content['product_name'].iloc[0]
similar_products = get_content_based_recommendations(test_product, 5)
print(f"\nContent-based recommendations for '{test_product}':")
for i, product in enumerate(similar_products, 1):
    print(f"  {i}. {product}")

print("\n Content-Based Filtering model created")


Content-based recommendations for 'Backpack':
  1. Watch
  2. Camera
  3. Charger
  4. Headphones
  5. Keyboard

 Content-Based Filtering model created


### Recommendation engine - Collaborative Filtering

In [79]:
# Create user-item matrix
user_item_matrix = df.pivot_table(index='customer_id', columns='product_name', 
                                   values='quantity', aggfunc='sum', fill_value=0)

print(f"User-Item matrix shape: {user_item_matrix.shape}")

User-Item matrix shape: (2000, 15)


In [85]:
# Calculate user similarity (cosine)
user_similarity = cosine_similarity(user_item_matrix)
user_similarity_df = pd.DataFrame(user_similarity, 
                                  index=user_item_matrix.index, 
                                  columns=user_item_matrix.index)

def get_user_based_recommendations(user_id, n_recommendations=5):
    """Get collaborative filtering recommendations for a user"""
    if user_id not in user_similarity_df.index:
        return get_popular_products(n_recommendations)
    
    # Find similar users
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)[1:11]
    
    # Get products purchased by similar users
    user_products = set(df[df['customer_id'] == user_id]['product_name'])
    recommendations = []
    for similar_user in similar_users.index:
        similar_user_products = set(df[df['customer_id'] == similar_user]['product_name'])
        new_products = similar_user_products - user_products
        recommendations.extend(list(new_products))
    
    # Count frequency of recommendations
    from collections import Counter
    recommendation_counts = Counter(recommendations)
    
    # Return top N recommendations
    top_recommendations = [product for product, count in 
                          recommendation_counts.most_common(n_recommendations)]
    
    return top_recommendations if top_recommendations else get_popular_products(n_recommendations)

def get_popular_products(n_recommendations=5):
    """Fallback: Return most popular products"""
    popular = product_features.nlargest(n_recommendations, 'popularity')['product_name'].tolist()
    return popular  

In [87]:
# Test collaborative filtering
test_user = df['customer_id'].iloc[0]
collab_recs = get_user_based_recommendations(test_user, 5)
print(f"\nCollaborative filtering recommendations for user '{test_user}':")
for i, product in enumerate(collab_recs, 1):
    print(f"  {i}. {product}")

print("\nCollaborative Filtering model created")


Collaborative filtering recommendations for user 'C100000':
  1. Monitor
  2. Tablet
  3. Backpack
  4. Charger
  5. Power Bank

Collaborative Filtering model created


### Recommendation Engine - Hybrid System

In [89]:
def get_hybrid_recommendations(user_id, n_recommendations=5, alpha=0.7):
    """
    Hybrid recommendations combining collaborative and content-based filtering
    alpha: weight for collaborative filtering (1-alpha for content-based)
    """
    # Get collaborative filtering recommendations
    collab_recs = get_user_based_recommendations(user_id, n_recommendations*2)
    
    # If no collab recs, return popular products
    if not collab_recs:
        return get_popular_products(n_recommendations)
    
    # For each collaborative recommendation, get similar products
    hybrid_scores = {}
    
    # Weight collaborative recommendations
    for i, product in enumerate(collab_recs):
        hybrid_scores[product] = hybrid_scores.get(product, 0) + alpha * (1 - i/len(collab_recs))
        
        # Add similar products with lower weight
        similar_products = get_content_based_recommendations(product, 3)
        for j, similar in enumerate(similar_products):
            hybrid_scores[similar] = hybrid_scores.get(similar, 0) + (1-alpha) * (1 - j/3)
    
    # Sort by score and return top N
    sorted_recs = sorted(hybrid_scores.items(), key=lambda x: x[1], reverse=True)
    final_recs = [product for product, score in sorted_recs[:n_recommendations]]
    
    return final_recs if final_recs else get_popular_products(n_recommendations)

In [91]:
# Test hybrid recommendations
print(f"\nHybrid recommendations for user '{test_user}':")
hybrid_recs = get_hybrid_recommendations(test_user, 5)
for i, product in enumerate(hybrid_recs, 1):
    print(f"  {i}. {product}")

print("\nHybrid Recommendation System created")


Hybrid recommendations for user 'C100000':
  1. Camera
  2. Charger
  3. Headphones
  4. Monitor
  5. Tablet

Hybrid Recommendation System created


### Generate Recommendation for all users 

In [101]:
### Generate Recommendations for all users with Category Information (Safe Version)

# First, create mappings
product_category_map = df[['product_name', 'category']].drop_duplicates().set_index('product_name')['category'].to_dict()
product_price_map = product_features.set_index('product_name')['avg_price'].to_dict()
product_priceband_map = product_features.set_index('product_name')['price_band'].to_dict()

In [121]:
# Generate recommendations
all_recommendations = []
for user_id in tqdm(customer_features['customer_id'].head(100)):
    recs = get_hybrid_recommendations(user_id, 5)
    for rank, product in enumerate(recs, 1):
        # Safely get values with defaults
        category = product_category_map.get(product, 'Unknown')
        price = product_price_map.get(product, 0)
        price_band = product_priceband_map.get(product, 'Unknown')
        
        recommendation = {
            'user_id': user_id,
            'rank': rank,
            'recommended_product': product,
            'category': category,
            'price_band': price_band,
            'avg_price': price,
            'recommendation_type': 'hybrid'
        }
        all_recommendations.append(recommendation)

100%|████████████████████████████████████████████████████████████████████████████████| 100/100 [00:01<00:00, 88.87it/s]


In [123]:
# Create DataFrame
recommendations_df = pd.DataFrame(all_recommendations)

In [125]:
# Define desired column order
desired_columns = ['user_id', 'rank', 'recommended_product', 'category', 'price_band', 'avg_price', 'recommendation_type']

In [127]:
# Check which columns actually exist in the dataframe
available_columns = recommendations_df.columns.tolist()
print("Available columns:", available_columns)


Available columns: ['user_id', 'rank', 'recommended_product', 'category', 'price_band', 'avg_price', 'recommendation_type']


In [129]:
# Only use columns that exist
final_columns = [col for col in desired_columns if col in available_columns]
recommendations_df = recommendations_df[final_columns]

In [131]:
# Sort
recommendations_df = recommendations_df.sort_values(['user_id', 'rank'])

In [133]:
# Save
recommendations_df.to_csv('recommended_products.csv', index=False)

print(f"\nSaved recommendations for {recommendations_df['user_id'].nunique()} users")
print(f"Total recommendations generated: {len(recommendations_df)}")

print("\nSample recommendations:")
print(recommendations_df.head(10).to_string(index=False))


Saved recommendations for 100 users
Total recommendations generated: 500

Sample recommendations:
user_id  rank recommended_product    category        price_band    avg_price recommendation_type
C100000     1              Camera Electronics Economy (10k-50k) 40457.568966              hybrid
C100000     2             Charger Electronics Economy (10k-50k) 40528.669065              hybrid
C100000     3          Headphones Electronics Economy (10k-50k) 37753.197080              hybrid
C100000     4             Monitor Electronics Economy (10k-50k) 39769.319728              hybrid
C100000     5              Tablet Electronics Economy (10k-50k) 42168.493151              hybrid
C100001     1              Camera Electronics Economy (10k-50k) 40457.568966              hybrid
C100001     2             Charger Electronics Economy (10k-50k) 40528.669065              hybrid
C100001     3          Headphones Electronics Economy (10k-50k) 37753.197080              hybrid
C100001     4             Mo

In [135]:
# Show statistics if category column exists
if 'category' in recommendations_df.columns:
    print("\nRecommendation distribution by category:")
    category_dist = recommendations_df['category'].value_counts()
    for category, count in category_dist.items():
        print(f"  {category}: {count} recommendations ({count/len(recommendations_df)*100:.1f}%)")
else:
    print("\nNote: Category information not available in the final DataFrame")
    print("Available columns:", recommendations_df.columns.tolist())

print("\n Step 2 Completed: Recommendations saved to 'recommended_products.csv'")


Recommendation distribution by category:
  Electronics: 500 recommendations (100.0%)

 Step 2 Completed: Recommendations saved to 'recommended_products.csv'


#### Sales Forcasting  - Data Preparation

In [137]:
# Prepare time series data
daily_sales = df.groupby(df['order_date'].dt.date)['total'].sum().reset_index()
daily_sales.columns = ['ds', 'y']
daily_sales['ds'] = pd.to_datetime(daily_sales['ds'])

print(f"Daily sales data: {len(daily_sales)} days")
print(f"Sales range: ₹{daily_sales['y'].min():,.2f} to ₹{daily_sales['y'].max():,.2f}")

Daily sales data: 365 days
Sales range: ₹13,469.85 to ₹2,191,620.55


## Create features for XGBoost

In [140]:
# Create features for XGBoost
daily_sales['day'] = daily_sales['ds'].dt.day
daily_sales['month'] = daily_sales['ds'].dt.month
daily_sales['year'] = daily_sales['ds'].dt.year
daily_sales['dayofweek'] = daily_sales['ds'].dt.dayofweek
daily_sales['quarter'] = daily_sales['ds'].dt.quarter
daily_sales['dayofyear'] = daily_sales['ds'].dt.dayofyear

print("\nDaily sales data with features:")
print(daily_sales.head())


Daily sales data with features:
          ds           y  day  month  year  dayofweek  quarter  dayofyear
0 2023-01-01   457536.60    1      1  2023          6        1          1
1 2023-01-02   398298.30    2      1  2023          0        1          2
2 2023-01-03   715716.60    3      1  2023          1        1          3
3 2023-01-04   402338.15    4      1  2023          2        1          4
4 2023-01-05  1010306.65    5      1  2023          3        1          5


#### Sales Forcasting - Create Lag Features

In [143]:
# Create lag features
for lag in [1, 2, 3, 7, 14]:
    daily_sales[f'lag_{lag}'] = daily_sales['y'].shift(lag)

In [145]:
# Create rolling statistics
for window in [3, 7, 14]:
    daily_sales[f'rolling_mean_{window}'] = daily_sales['y'].rolling(window=window).mean()
    daily_sales[f'rolling_std_{window}'] = daily_sales['y'].rolling(window=window).std()

In [147]:
# Drop NaN values
daily_sales_clean = daily_sales.dropna()

print(f"Shape after creating features: {daily_sales_clean.shape}")
print("\nFeatures created:")
feature_cols = ['day', 'month', 'year', 'dayofweek', 'quarter', 'dayofyear',
                'lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_14',
                'rolling_mean_3', 'rolling_mean_7', 'rolling_mean_14',
                'rolling_std_3', 'rolling_std_7', 'rolling_std_14']
print(feature_cols)

Shape after creating features: (351, 19)

Features created:
['day', 'month', 'year', 'dayofweek', 'quarter', 'dayofyear', 'lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_14', 'rolling_mean_3', 'rolling_mean_7', 'rolling_mean_14', 'rolling_std_3', 'rolling_std_7', 'rolling_std_14']


### Train XGBoost Model


### Prepare features for training

In [151]:
# Prepare features for training
X = daily_sales_clean[feature_cols]
y = daily_sales_clean['y']

In [153]:
# Split data
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Training data size: {len(X_train)} days")
print(f"Testing data size: {len(X_test)} days")


Training data size: 280 days
Testing data size: 71 days


In [155]:
# Train XGBoost model
print("\nTraining XGBoost model...")
xgb_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
xgb_model.fit(X_train, y_train)


Training XGBoost model...


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=5, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=100, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

In [157]:
# Make predictions
train_pred = xgb_model.predict(X_train)
test_pred = xgb_model.predict(X_test)

In [159]:
# Calculate metrics
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
train_mae = mean_absolute_error(y_train, train_pred)
test_mae = mean_absolute_error(y_test, test_pred)

print(f"\nModel Performance:")
print(f"  Train RMSE: ₹{train_rmse:,.2f}")
print(f"  Test RMSE: ₹{test_rmse:,.2f}")
print(f"  Train MAE: ₹{train_mae:,.2f}")
print(f"  Test MAE: ₹{test_mae:,.2f}")


Model Performance:
  Train RMSE: ₹7,310.30
  Test RMSE: ₹119,133.88
  Train MAE: ₹5,597.11
  Test MAE: ₹93,075.92


### Generate Future Forecasts

In [162]:
# Prepare future dates
last_date = daily_sales['ds'].max()
future_dates = [last_date + timedelta(days=i) for i in range(1, 91)]

In [164]:
# Create future dataframe
future_df = pd.DataFrame({'ds': future_dates})
future_df['day'] = future_df['ds'].dt.day
future_df['month'] = future_df['ds'].dt.month
future_df['year'] = future_df['ds'].dt.year
future_df['dayofweek'] = future_df['ds'].dt.dayofweek
future_df['quarter'] = future_df['ds'].dt.quarter
future_df['dayofyear'] = future_df['ds'].dt.dayofyear


In [166]:
# Use last known values for lag features
last_known = daily_sales.iloc[-1]
for lag in [1, 2, 3, 7, 14]:
    future_df[f'lag_{lag}'] = last_known[f'lag_{lag}']

for window in [3, 7, 14]:
    future_df[f'rolling_mean_{window}'] = last_known[f'rolling_mean_{window}']
    future_df[f'rolling_std_{window}'] = last_known[f'rolling_std_{window}']


In [168]:
# Generate forecasts
future_predictions = xgb_model.predict(future_df[feature_cols])


In [170]:
# Create forecast dataframe
forecast_df = pd.DataFrame({
    'date': future_dates,
    'forecasted_sales': future_predictions,
    'forecast_period': [f'Day {i}' for i in range(1, 91)]
})


In [172]:
# Add confidence intervals (simplified)
forecast_df['lower_bound'] = forecast_df['forecasted_sales'] * 0.85
forecast_df['upper_bound'] = forecast_df['forecasted_sales'] * 1.15

### Forecast Summary

In [175]:
# Aggregate by period
forecast_summary = pd.DataFrame({
    'period': ['30 Days', '60 Days', '90 Days'],
    'total_forecasted_sales': [
        future_predictions[:30].sum(),
        future_predictions[:60].sum(),
        future_predictions[:90].sum()
    ],
    'avg_daily_sales': [
        future_predictions[:30].mean(),
        future_predictions[:60].mean(),
        future_predictions[:90].mean()
    ]
})

print("\nForecast Summary:")
print(forecast_summary.to_string(index=False))


Forecast Summary:
 period  total_forecasted_sales  avg_daily_sales
30 Days              16920036.0      564001.1875
60 Days              33513564.0      558559.3750
90 Days              50073204.0      556368.9375


In [177]:
# Save forecast
forecast_df.to_csv('sales_forecast.csv', index=False)
print("\nSaved forecast to 'sales_forecast.csv'")

print("\nSample forecast data (first 10 days):")
print(forecast_df.head(10))


Saved forecast to 'sales_forecast.csv'

Sample forecast data (first 10 days):
        date  forecasted_sales forecast_period   lower_bound  upper_bound
0 2024-01-02       568797.1250           Day 1  483477.56250  654116.6875
1 2024-01-03       568797.1250           Day 2  483477.56250  654116.6875
2 2024-01-04       568797.1250           Day 3  483477.56250  654116.6875
3 2024-01-05       568797.1250           Day 4  483477.56250  654116.6875
4 2024-01-06       565707.6875           Day 5  480851.56250  650563.8125
5 2024-01-07       565707.6875           Day 6  480851.56250  650563.8125
6 2024-01-08       566353.2500           Day 7  481400.28125  651306.2500
7 2024-01-09       569980.3750           Day 8  484483.34375  655477.4375
8 2024-01-10       569980.3750           Day 9  484483.34375  655477.4375
9 2024-01-11       569980.3750          Day 10  484483.34375  655477.4375


### Customer Segmentation- RFM Scoring

In [180]:
# Use RFM scores for segmentation
rfm_data = customer_features.copy()

In [182]:
# Calculate RFM scores (1-5 scale)
rfm_data['recency_score'] = pd.qcut(rfm_data['recency'], 5, labels=[5, 4, 3, 2, 1])
rfm_data['frequency_score'] = pd.qcut(rfm_data['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])
rfm_data['monetary_score'] = pd.qcut(rfm_data['monetary'], 5, labels=[1, 2, 3, 4, 5])

In [184]:
# Convert to numeric
rfm_data['recency_score'] = rfm_data['recency_score'].astype(int)
rfm_data['frequency_score'] = rfm_data['frequency_score'].astype(int)
rfm_data['monetary_score'] = rfm_data['monetary_score'].astype(int)

In [186]:
# Calculate RFM score
rfm_data['rfm_score'] = rfm_data['recency_score'] + rfm_data['frequency_score'] + rfm_data['monetary_score']

print("\nRFM Score Distribution:")
print(rfm_data['rfm_score'].describe())

print("\nSample RFM data:")
print(rfm_data[['customer_id', 'recency', 'frequency', 'monetary', 'rfm_score']].head())


RFM Score Distribution:
count    2000.000000
mean        9.005000
std         2.468405
min         3.000000
25%         7.000000
50%         9.000000
75%        11.000000
max        15.000000
Name: rfm_score, dtype: float64

Sample RFM data:
  customer_id  recency  frequency   monetary  rfm_score
0     C100000      225          1  100684.00          6
1     C100001      193          1   65983.20          7
2     C100002      102          1  259411.75         10
3     C100003      160          1   24071.10          5
4     C100004      173          1   32973.20          6


### K-Means Clustering

In [189]:
# Prepare features for clustering
cluster_features = ['recency', 'frequency', 'monetary', 'avg_quantity', 'avg_unit_price']
X_cluster = rfm_data[cluster_features].copy()

In [191]:
# Normalize features
scaler = StandardScaler()
X_cluster_scaled = scaler.fit_transform(X_cluster)


In [193]:
# Determine optimal number of clusters using elbow method
inertias = []
K_range = range(2, 9)
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_cluster_scaled)
    inertias.append(kmeans.inertia_)

print("\nElbow Method Results:")
for k, inertia in zip(K_range, inertias):
    print(f"  K={k}: Inertia={inertia:.2f}")


Elbow Method Results:
  K=2: Inertia=5304.68
  K=3: Inertia=4167.28
  K=4: Inertia=3437.45
  K=5: Inertia=2930.92
  K=6: Inertia=2508.93
  K=7: Inertia=2211.65
  K=8: Inertia=1957.34


In [194]:
# Choose k=5 for business-interpretable segments
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
rfm_data['cluster'] = kmeans.fit_predict(X_cluster_scaled)

print("\n K-Means clustering completed with 5 clusters")


 K-Means clustering completed with 5 clusters


### Customer Segmentation - Assign Business Labels

In [198]:
# Analyze clusters and assign business labels
cluster_profiles = rfm_data.groupby('cluster').agg({
    'recency': 'mean',
    'frequency': 'mean',
    'monetary': 'mean',
    'rfm_score': 'mean'
}).round(2)

print("\nCluster Profiles:")
print(cluster_profiles)


Cluster Profiles:
         recency  frequency   monetary  rfm_score
cluster                                          
0         156.44        1.0   94273.37       9.48
1          74.41        1.0   57975.84       9.50
2         264.02        1.0   96451.48       7.92
3         258.60        1.0   28068.46       6.43
4         178.92        1.0  240464.64      10.91


In [200]:
# Define business-friendly labels based on cluster characteristics
segment_labels = {
    0: 'High-Value Loyal Customers',
    1: 'Potential Loyalists',
    2: 'At-Risk Customers',
    3: 'New Buyers',
    4: 'Price-Sensitive Shoppers'
}


In [202]:
# Add labels to the data
rfm_data['segment'] = rfm_data['cluster'].map(segment_labels)


In [204]:
# Get segment distribution
segment_distribution = rfm_data['segment'].value_counts()
print("\nCustomer Segment Distribution:")
print(segment_distribution)


Customer Segment Distribution:
segment
Price-Sensitive Shoppers      458
High-Value Loyal Customers    430
Potential Loyalists           409
At-Risk Customers             356
New Buyers                    347
Name: count, dtype: int64


In [206]:
# Segment characteristics
segment_analysis = rfm_data.groupby('segment').agg({
    'customer_id': 'count',
    'monetary': ['mean', 'sum'],
    'frequency': 'mean',
    'recency': 'mean'
}).round(2)

print("\nSegment Analysis:")
print(segment_analysis)


Segment Analysis:
                           customer_id   monetary               frequency  \
                                 count       mean           sum      mean   
segment                                                                     
At-Risk Customers                  356   96451.48  3.433673e+07       1.0   
High-Value Loyal Customers         430   94273.37  4.053755e+07       1.0   
New Buyers                         347   28068.46  9.739756e+06       1.0   
Potential Loyalists                409   57975.84  2.371212e+07       1.0   
Price-Sensitive Shoppers           458  240464.64  1.101328e+08       1.0   

                           recency  
                              mean  
segment                             
At-Risk Customers           264.02  
High-Value Loyal Customers  156.44  
New Buyers                  258.60  
Potential Loyalists          74.41  
Price-Sensitive Shoppers    178.92  


### Customer Segmentation - Save Results

In [209]:
# Prepare customer segmentation output
customer_segments = rfm_data[['customer_id', 'recency', 'frequency', 'monetary', 
                              'rfm_score', 'segment', 'cluster']].copy()
customer_segments.to_csv('customer_segments.csv', index=False)

print("\n Saved customer segments to 'customer_segments.csv'")
print("\nSample of final customer segments:")
print(customer_segments.head(10))

print("\nStep 3 Completed: Sales Forecast and Customer Segmentation")


 Saved customer segments to 'customer_segments.csv'

Sample of final customer segments:
  customer_id  recency  frequency   monetary  rfm_score  \
0     C100000      225          1  100684.00          6   
1     C100001      193          1   65983.20          7   
2     C100002      102          1  259411.75         10   
3     C100003      160          1   24071.10          5   
4     C100004      173          1   32973.20          6   
5     C100005      145          1  290920.00          9   
6     C100006      108          1   41705.25          7   
7     C100007      220          1  108085.00          6   
8     C100008      330          1  160076.25          6   
9     C100009      319          1    7219.05          3   

                      segment  cluster  
0           At-Risk Customers        2  
1           At-Risk Customers        2  
2    Price-Sensitive Shoppers        4  
3                  New Buyers        3  
4                  New Buyers        3  
5    Price-Sens

## Key Metrics

In [212]:
# Prepare sales overview data
sales_overview = {
    'total_revenue': df['total'].sum(),
    'total_orders': df['order_id'].nunique(),
    'total_customers': df['customer_id'].nunique(),
    'total_products': df['product_name'].nunique(),
    'avg_order_value': df.groupby('order_id')['total'].sum().mean(),
    'avg_customer_value': df.groupby('customer_id')['total'].sum().mean()
}

print("\nSales Overview:")
for key, value in sales_overview.items():
    if 'revenue' in key or 'value' in key:
        print(f"  {key}: ₹{value:,.2f}")
    else:
        print(f"  {key}: {value:,}")


Sales Overview:
  total_revenue: ₹218,458,955.65
  total_orders: 2,000
  total_customers: 2,000
  total_products: 15
  avg_order_value: ₹109,229.48
  avg_customer_value: ₹109,229.48


## Top Products and Categories

In [215]:
# Top products for cross-sell insights
top_products = product_features.nlargest(10, 'total_revenue')[['product_name', 'category', 'total_revenue', 'popularity']]
print("\nTop 10 Products by Revenue:")
print(top_products.to_string(index=False))


Top 10 Products by Revenue:
product_name    category  total_revenue  popularity
      Tablet Electronics    16658066.15         146
     Charger Electronics    16316790.80         139
  Power Bank Electronics    15642077.15         139
      Laptop Electronics    15532507.55         132
       Mouse Electronics    15507647.95         136
       Shoes     Fashion    15302155.35         137
     Monitor Electronics    15156792.35         147
       Jeans     Fashion    15066470.35         137
    Backpack Accessories    14038257.85         139
  Headphones Electronics    13911314.55         137


In [217]:
# Category performance
category_performance = df.groupby('category').agg({
    'total': 'sum',
    'order_id': 'count',
    'customer_id': 'nunique'
}).reset_index()
category_performance.columns = ['category', 'revenue', 'orders', 'unique_customers']
category_performance['revenue_percent'] = category_performance['revenue'] / category_performance['revenue'].sum() * 100

print("\nCategory Performance:")
print(category_performance.to_string(index=False))


Category Performance:
   category      revenue  orders  unique_customers  revenue_percent
Accessories  27878602.20     270               270        12.761483
Electronics 147085718.45    1338              1338        67.328766
    Fashion  43494635.00     392               392        19.909751


## Monthly Trends and Cross-Sell

In [220]:
# Monthly trends
monthly_trends = df.groupby(df['order_date'].dt.to_period('M'))['total'].agg(['sum', 'count', 'mean']).reset_index()
monthly_trends.columns = ['month', 'revenue', 'order_count', 'avg_order_value']
monthly_trends['month'] = monthly_trends['month'].astype(str)

print("\nMonthly Trends (last 3 months):")
print(monthly_trends.tail(3).to_string(index=False))


Monthly Trends (last 3 months):
  month     revenue  order_count  avg_order_value
2023-11 18540790.95          159    116608.748113
2023-12 19101211.85          175    109149.782000
2024-01   515258.70            4    128814.675000


In [222]:
# Prepare cross-sell recommendations
cross_sell_pairs = []
for product in top_products['product_name'].head(5):
    similar = get_content_based_recommendations(product, 3)
    for similar_product in similar:
        cross_sell_pairs.append({
            'product': product,
            'cross_sell_product': similar_product,
            'similarity_score': 0.85  # Placeholder
        })

cross_sell_df = pd.DataFrame(cross_sell_pairs)
cross_sell_df.to_csv('cross_sell_insights.csv', index=False)

print("\n Cross-sell insights saved to 'cross_sell_insights.csv'")
print("\ncross-sell pairs:")
print(cross_sell_df.head())


 Cross-sell insights saved to 'cross_sell_insights.csv'

cross-sell pairs:
   product cross_sell_product  similarity_score
0   Tablet             Camera              0.85
1   Tablet            Charger              0.85
2   Tablet         Headphones              0.85
3  Charger             Camera              0.85
4  Charger         Headphones              0.85


## Integration-Ready Output

In [225]:
# Prepare all final CSV files
# Read saved files to verify
final_recommendations = pd.read_csv('recommended_products.csv')
final_segments = pd.read_csv('customer_segments.csv')
final_forecast = pd.read_csv('sales_forecast.csv')
final_cross_sell = pd.read_csv('cross_sell_insights.csv')

print(f"\nFiles ready for integration:")
print(f"  - recommended_products.csv: {len(final_recommendations)} recommendations")
print(f"  - customer_segments.csv: {len(final_segments)} customers segmented")
print(f"  - sales_forecast.csv: {len(final_forecast)} days forecasted")
print(f"  - cross_sell_insights.csv: {len(final_cross_sell)} cross-sell pairs")
print(f"  - cleaned_sales_data.csv: {len(df_cleaned)} records")


Files ready for integration:
  - recommended_products.csv: 500 recommendations
  - customer_segments.csv: 2000 customers segmented
  - sales_forecast.csv: 90 days forecasted
  - cross_sell_insights.csv: 15 cross-sell pairs
  - cleaned_sales_data.csv: 2000 records


## Integration-Ready Output - API Documentation

In [228]:
# Prepare API structure (documentation)
print("\n5.2 API Documentation:")
print("""
REST API Endpoint Structure:
---------------------------
GET /api/recommendations?user_id={user_id}&n={count}
   - Returns top N product recommendations for a user
   - Response format: JSON with product details
   - Sample: /api/recommendations?user_id=C100000&n=5

GET /api/customer/segment?user_id={user_id}
   - Returns customer segment information
   - Includes RFM scores and segment label
   - Sample: /api/customer/segment?user_id=C100000

GET /api/products/similar?product_id={product_id}
   - Returns similar products for cross-selling
   - Sample: /api/products/similar?product_id=Smartphone

GET /api/forecast/sales?days={30/60/90}
   - Returns sales forecast for specified period
   - Sample: /api/forecast/sales?days=30

Authentication: Token-based (API key required)
Rate Limiting: 100 requests per minute per key
""")


5.2 API Documentation:

REST API Endpoint Structure:
---------------------------
GET /api/recommendations?user_id={user_id}&n={count}
   - Returns top N product recommendations for a user
   - Response format: JSON with product details
   - Sample: /api/recommendations?user_id=C100000&n=5

GET /api/customer/segment?user_id={user_id}
   - Returns customer segment information
   - Includes RFM scores and segment label
   - Sample: /api/customer/segment?user_id=C100000

GET /api/products/similar?product_id={product_id}
   - Returns similar products for cross-selling
   - Sample: /api/products/similar?product_id=Smartphone

GET /api/forecast/sales?days={30/60/90}
   - Returns sales forecast for specified period
   - Sample: /api/forecast/sales?days=30

Authentication: Token-based (API key required)
Rate Limiting: 100 requests per minute per key

